# 02-chunk · 02 — token-aware chunking (Docling `HybridChunker`)

**The real gain here is structural: a chunk boundary can no longer land inside a table, because `HybridChunker` walks Docling's layout tree instead of counting words.**

This notebook implements the more capable of the two chunking strategies in
this stage: structure-aware chunking built on Docling's `HybridChunker`
instead of a flat-text splitter. It builds up in the same order the code
itself depends on: construct the chunker, look at its raw output, then add
the table-detection and chunk-type-tagging logic on top of that raw output —
never the other way around.

**Implements:** the table/figure/prose tagging (`get_chunk_type`), the table
markdown/structured/cell extraction helpers, the `HybridChunker`
construction, and the `MIN_CHUNK_WORDS` floor that keeps a bare heading like
`"Article"` from becoming its own embedded chunk.

**Out of scope for this notebook:** the local `BAAI/bge-large-en-v1.5`
`SentenceTransformer` embedding call — deciding the embedding model is stage
03's job, so `embed_fn` below is a clearly marked stub. No document-set or
domain-specific configuration is baked in here either; this notebook works
against whatever documents you point it at.

**Adaptation note (version drift):** in the installed `docling-core`
(2.86.0), `chunk.meta.doc_items` returns lightweight item stubs — they carry
`.label` and `.self_ref` but not the `TableItem` subclass methods
(`export_to_markdown`, `.data`) needed for table rendering. This notebook
resolves each item back to the fully-typed object via `doc.tables[i]` /
`doc.pictures[i]`, keyed by the index encoded in `self_ref` (e.g.
`"#/tables/0"`). This gap is demonstrated directly (Step 4) before the shim
that fixes it (Step 5) — the drift is kept documented, not silently patched
away.

## What this notebook demonstrates

| Name | What it does | Example |
| --- | --- | --- |
| `build_hybrid_chunker()` | Constructs a `HybridChunker` with table-header repetition, peer merging, and heading emission turned on | `build_hybrid_chunker().chunk(doc)` |
| `_item_label(item)` | Reads an item's label off either a stub or a fully-typed object | `_item_label(stub) -> "table"` |
| `_resolve_doc_item(item, doc)` | Resolves a `doc_items` stub back to the fully-typed `TableItem`/`PictureItem` (the version-drift shim) | `_resolve_doc_item(stub, doc)` -> real `TableItem` |
| `_try_table_markdown(doc_items, doc)` | Extracts a table's markdown export via Docling's native `export_to_markdown` | returns the 5-row outcomes table as markdown |
| `_try_table_structured(doc_items, doc)` | Extracts a table as a structured, DataFrame-shaped dict | `{"headers": [...], "rows": [...], ...}` |
| `_extract_table_cells(doc_items, doc)` | Extracts cell-level metadata with bounding boxes and header flags | list of per-cell dicts |
| `get_chunk_type(chunk_obj, doc)` | Tags a chunk `"table"`, `"figure"`, or `"prose"` from its resolved `doc_items` labels | `get_chunk_type(c, doc) -> "table"` |
| `chunk_document_to_records(doc, doc_id)` | Runs the full pipeline: chunk, tag, extract tables, apply the `MIN_CHUNK_WORDS` floor, build the chunk-record contract | 2 records for the synthetic outcomes paper |
| `embed_fn_stub(texts)` | Deliberately unimplemented placeholder — embedding is stage 03's job | raises `NotImplementedError` |


In [1]:
import sys
from pathlib import Path

# The kernel's cwd is this notebook's own directory (that's how Jupyter
# starts kernels), not the repo root -- so a bare `import nbio` fails two
# directories down unless the repo root goes on sys.path first. Same
# walk-up nbio.py's own bootstrap() uses internally.
_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()


WindowsPath('C:/Users/saima/Documents/anacodic-agentic-cookbook')

## Step 1 — build the synthetic outcomes document through Docling's markdown backend

`HybridChunker` needs a structured `DoclingDocument`, not a raw string.
Rather than write a separate table-construction API, we hand the identical
prose + table content used in notebook `01` to Docling's markdown backend —
it parses `##` headings into a heading tree and pipe tables into real
`TableItem` objects, entirely locally (no model download, no network call;
markdown parsing does not touch the layout/OCR models that PDF ingestion
needs). This is also how the extract stage's output would arrive here in a
full pipeline run: as a `DoclingDocument`, already structured.

In [2]:
import tempfile
from pathlib import Path
from docling.document_converter import DocumentConverter

prose_sentence = (
    "Patients were followed for twelve months after the index procedure to "
    "record wound healing time and complication rates in each treatment arm. "
)
prose = (prose_sentence * 6).strip()

synthetic_md = f"""# Results

{prose}

## Outcomes Table

| Patient ID | Treatment Arm | Wound Healing (days) | Complication |
| --- | --- | --- | --- |
| P001 | Early excision | 14 | None |
| P002 | Delayed excision | 21 | Infection |
| P003 | Early excision | 12 | None |
| P004 | Delayed excision | 25 | Graft failure |
| P005 | Early excision | 15 | None |

These results are discussed further in the next section.
"""

tmp_path = Path(tempfile.mkstemp(suffix=".md")[1])
tmp_path.write_text(synthetic_md)

doc = DocumentConverter().convert(str(tmp_path)).document
print(doc.export_to_markdown()[:300])


c:\Users\saima\Documents\anacodic-agentic-cookbook\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Results

Patients were followed for twelve months after the index procedure to record wound healing time and complication rates in each treatment arm. Patients were followed for twelve months after the index procedure to record wound healing time and complication rates in each treatment arm. Patie


## Step 2 — construct the `HybridChunker`

Structure-aware: it walks the document's layout tree (headings, paragraphs,
tables, figures) rather than a flat string, so a chunk boundary can never
land inside a table the way it did in notebook `01`. `repeat_table_header=True`
and `merge_peers=True` keep a table's header attached to every emitted piece
of it and merge adjacent same-type chunks; `always_emit_headings=True` keeps
the section breadcrumb (`section_path`, added later) attached to every chunk.

`MIN_CHUNK_WORDS` exists because `HybridChunker` emits structural units, so a
bare heading or stray label can otherwise arrive as a one-word "chunk" (e.g.
`text: "Article"` reaching a real index). Tables and figures are exempt,
since they are legitimately terse — that floor is applied later, in Step 10.

In [3]:
from docling.chunking import HybridChunker

MIN_CHUNK_WORDS = 25


def build_hybrid_chunker():
    return HybridChunker(
        repeat_table_header=True,
        merge_peers=True,
        always_emit_headings=True,
    )


## Step 3 — run the chunker and look at the raw chunks

Before any table-detection or type-tagging logic exists, look at what
`HybridChunker` actually emits for the synthetic document: how many chunks,
how many `doc_items` each carries, and a text preview. Everything below this
point is built on top of what's printed here.

In [4]:
chunker = build_hybrid_chunker()
raw_chunks = list(chunker.chunk(doc))
print(f"{len(raw_chunks)} raw chunks from HybridChunker\n")
for i, c in enumerate(raw_chunks):
    n_items = len(getattr(getattr(c, "meta", None), "doc_items", None) or [])
    text_preview = (getattr(c, "text", "") or "")[:100].replace("\n", " ")
    print(f"chunk {i}: {n_items} doc_items, text[:100]={text_preview!r}")


2 raw chunks from HybridChunker

chunk 0: 1 doc_items, text[:100]='Patients were followed for twelve months after the index procedure to record wound healing time and '
chunk 1: 2 doc_items, text[:100]='P001, Treatment Arm = Early excision. P001, Wound Healing (days) = 14. P001, Complication = None. P0'


## Step 4 — the docling-core version-drift gap

`_item_label` reads an item's label off either a stub or a fully-typed
object. Use it to find the table chunk's `doc_items` entry straight out of
`raw_chunks` above, with no resolution applied yet — and look at what's
missing: the label reads fine, but the subclass method table rendering needs
(`export_to_markdown`) isn't there. This is the version-drift gap named in
the intro, shown directly rather than patched silently.

In [5]:
from typing import Any


def _item_label(item: Any) -> str:
    label = getattr(item, "label", None)
    return str(getattr(label, "value", label) or "").lower()


table_item_stub = None
for c in raw_chunks:
    for it in (getattr(getattr(c, "meta", None), "doc_items", None) or []):
        if _item_label(it) == "table":
            table_item_stub = it
            break
    if table_item_stub is not None:
        break

print(f"stub label: {_item_label(table_item_stub)!r}")
print(f"stub type: {type(table_item_stub).__name__}")
print(f"has export_to_markdown: {hasattr(table_item_stub, 'export_to_markdown')}")


stub label: 'table'
stub type: DocItem
has export_to_markdown: False


## Step 5 — resolve the stub back to the typed object (the version-drift shim)

`_resolve_doc_item` maps a `doc_items` stub back to the real `TableItem` /
`PictureItem`, keyed by the index encoded in `self_ref` (e.g.
`"#/tables/0"`). Run it on the exact stub from Step 4 and confirm the
subclass method is now present.

In [6]:
def _resolve_doc_item(item: Any, doc: Any) -> Any:
    """chunk.meta.doc_items entries are reference stubs in this docling-core
    version; resolve back to the real TableItem/PictureItem so subclass
    methods like export_to_markdown() are available. See the version-drift
    note above."""
    ref = getattr(item, "self_ref", "") or ""
    try:
        if ref.startswith("#/tables/"):
            return doc.tables[int(ref.rsplit("/", 1)[-1])]
        if ref.startswith("#/pictures/"):
            return doc.pictures[int(ref.rsplit("/", 1)[-1])]
    except (IndexError, ValueError):
        pass
    return item


resolved_table_item = _resolve_doc_item(table_item_stub, doc)
print(f"resolved type: {type(resolved_table_item).__name__}")
print(f"has export_to_markdown: {hasattr(resolved_table_item, 'export_to_markdown')}")


resolved type: TableItem
has export_to_markdown: True


## Step 6 — table extraction: markdown export

`_try_table_markdown` walks a chunk's `doc_items`, resolves each one, and
returns the first table's native markdown export — this is both the display
form and (after notebook 02's own override below) the text actually sent to
an embedder. Test it on the table chunk's full `doc_items` list.

In [ ]:
def _try_table_markdown(doc_items: list[Any], doc: Any) -> str:
    """Extract markdown format from table items using Docling's native export."""
    for raw in doc_items:
        item = _resolve_doc_item(raw, doc)
        if _item_label(item) == "table" and hasattr(item, "export_to_markdown"):
            try:
                return item.export_to_markdown(doc=doc)
            except Exception:
                pass
    return ""


table_chunk_doc_items = next(
    getattr(getattr(c, "meta", None), "doc_items", None) or []
    for c in raw_chunks
    if any(_item_label(it) == "table" for it in (getattr(getattr(c, "meta", None), "doc_items", None) or []))
)
table_markdown = _try_table_markdown(table_chunk_doc_items, doc)
print(table_markdown)


## Step 7 — table extraction: structured (DataFrame-shaped) export

`_try_table_structured` returns the same table as a plain dict of headers and
rows — useful for anything downstream that wants columns, not markdown
text.

In [ ]:
def _try_table_structured(doc_items: list[Any], doc: Any) -> "dict[str, Any] | None":
    """Extract structured table data (DataFrame representation)."""
    for raw in doc_items:
        item = _resolve_doc_item(raw, doc)
        if _item_label(item) == "table" and hasattr(item, "data"):
            try:
                if hasattr(item, "export_to_dataframe"):
                    df = item.export_to_dataframe(doc=doc)
                    return {
                        "headers": df.columns.tolist() if hasattr(df, "columns") else [],
                        "rows": df.to_dict("records") if hasattr(df, "to_dict") else [],
                        "num_rows": len(df) if hasattr(df, "__len__") else 0,
                        "num_cols": len(df.columns) if hasattr(df, "columns") else 0,
                    }
            except Exception:
                pass
    return None


structured = _try_table_structured(table_chunk_doc_items, doc)
print(f"headers: {structured['headers']}")
print(f"num_rows={structured['num_rows']}  num_cols={structured['num_cols']}")
for row in structured["rows"]:
    print(row)


## Step 8 — table extraction: cell-level export with bounding boxes

`_extract_table_cells` goes one level deeper than the structured export:
per-cell text, row/col position, span, header flag, and bounding box —
the level of detail a layout-aware renderer would need.

In [ ]:
def _extract_table_cells(doc_items: list[Any], doc: Any) -> "list[dict[str, Any]] | None":
    """Extract cell-level metadata with bounding boxes."""
    for raw in doc_items:
        item = _resolve_doc_item(raw, doc)
        if _item_label(item) == "table" and hasattr(item, "data"):
            try:
                table_data = item.data
                if not hasattr(table_data, "table_cells"):
                    return None
                cells = []
                for cell in table_data.table_cells:
                    cell_dict = {
                        "text": str(getattr(cell, "text", "")),
                        "row": getattr(cell, "start_row_offset_idx", 0),
                        "col": getattr(cell, "start_col_offset_idx", 0),
                        "row_span": getattr(cell, "row_span", 1),
                        "col_span": getattr(cell, "col_span", 1),
                        "is_header": getattr(cell, "column_header", False) or getattr(cell, "row_header", False),
                    }
                    if hasattr(cell, "bbox"):
                        bbox = cell.bbox
                        cell_dict["bbox"] = {
                            "l": getattr(bbox, "l", 0), "t": getattr(bbox, "t", 0),
                            "r": getattr(bbox, "r", 0), "b": getattr(bbox, "b", 0),
                        }
                    cells.append(cell_dict)
                return cells if cells else None
            except Exception:
                pass
    return None


cells = _extract_table_cells(table_chunk_doc_items, doc)
print(f"{len(cells)} cells\n")
for cell in cells[:4]:
    print(cell)


## Step 9 — tag each chunk's type

`get_chunk_type` ties the label + resolution helpers together into the
one-word tag (`"table"`, `"figure"`, or `"prose"`) the rest of the pipeline
keys off of. Run it over every raw chunk from Step 3.

In [ ]:
def get_chunk_type(chunk_obj: Any, doc: Any) -> str:
    try:
        doc_items = list(getattr(getattr(chunk_obj, "meta", None), "doc_items", []) or [])
        labels = {_item_label(_resolve_doc_item(it, doc)) for it in doc_items}
        if "table" in labels:
            return "table"
        if labels & {"picture", "figure"}:
            return "figure"
    except Exception:
        pass
    return "prose"


for i, c in enumerate(raw_chunks):
    print(f"chunk {i}: type={get_chunk_type(c, doc)}")


## Step 10 — `_tail_by_tokens`, the overlap `HybridChunker` doesn't have natively

Ported verbatim from `docling/src/paper_extract/chunking.py`, including its docstring — the reason it exists is easy to miss and easy to delete as "redundant" without it: **`HybridChunker` has no native overlap between chunks.** Each chunk is exactly the text `HybridChunker` assigned it, nothing more. `_tail_by_tokens` reconstructs overlap after the fact by walking the *previous* chunk's own words from the end, growing the suffix, and stopping as soon as the same tokenizer that sizes chunks says it has enough — word granularity, so the prepended text starts on a word boundary, not mid-word.

In [ ]:
def _tail_by_tokens(text: str, tokenizer, min_tokens: int) -> str:
    """The shortest word-suffix of *text* whose token count is >= *min_tokens*.

    HybridChunker has no native overlap between chunks, so this reconstructs it:
    walk the previous chunk's words from the end, growing the suffix, and stop as
    soon as the tokenizer (the same one sizing chunks) says it has enough. Word
    granularity, not character, so the prepended text starts on a word boundary.
    """
    words = text.split()
    if not words or min_tokens <= 0:
        return ""
    for k in range(1, len(words) + 1):
        candidate = " ".join(words[-k:])
        if tokenizer.count_tokens(candidate) >= min_tokens:
            return candidate
    return " ".join(words)


# look at real output before wiring this into the record assembly below:
chunker_for_demo = build_hybrid_chunker()
demo_tail = _tail_by_tokens(raw_chunks[0].text, chunker_for_demo.tokenizer, min_tokens=15)
print(f"last ~15 tokens of chunk 0, by word-suffix growth:\n{demo_tail!r}")

## Step 11 — assemble the full chunk-record contract

`chunk_document_to_records` ties every piece above together: chunk, tag with `get_chunk_type`, override a table chunk's text with its markdown export, prepend the previous chunk's tail when `overlap > 0` (Step 10), and stamp `is_table`/`is_figure` booleans alongside the existing `chunk_type` string -- a downstream reader can check either one; they're the same fact, in the two shapes docling's original and this cookbook's contract each chose.

In [ ]:
def chunk_document_to_records(
    doc, doc_id: str, min_chunk_words: int = MIN_CHUNK_WORDS, overlap: int = 0
) -> list[dict]:
    chunker = build_hybrid_chunker()
    raw_chunks = list(chunker.chunk(doc))
    records = []
    kept = 0
    for idx, c in enumerate(raw_chunks):
        ctype = get_chunk_type(c, doc)
        raw_text = getattr(c, "text", "") or ""
        embed_text = chunker.contextualize(c) or raw_text

        doc_items = list(getattr(getattr(c, "meta", None), "doc_items", None) or [])
        if ctype == "table":
            markdown_text = _try_table_markdown(doc_items, doc)
            if markdown_text:
                raw_text = markdown_text
                embed_text = markdown_text

        # Prose below the floor is a heading or label, not evidence. Tables
        # and figures are exempt (legitimately terse).
        if ctype == "prose" and len(raw_text.split()) < min_chunk_words:
            continue

        # Overlap looks at the PREVIOUS raw chunk's own text (before any
        # table-markdown override), matching docling's chunk_document(): the
        # point is continuity with what HybridChunker actually emitted next
        # to this one, not with whichever earlier records survived the
        # min_chunk_words filter above.
        if overlap > 0 and idx > 0:
            prev_text = getattr(raw_chunks[idx - 1], "text", "") or ""
            prefix = _tail_by_tokens(prev_text, chunker.tokenizer, overlap)
            if prefix:
                embed_text = f"{prefix}\n\n{embed_text}"

        headings = getattr(getattr(c, "meta", None), "headings", None) or []
        pages: set[int] = set()
        for item in doc_items:
            for prov in getattr(item, "prov", None) or []:
                try:
                    pages.add(int(prov.page_no))
                except (TypeError, ValueError):
                    pass

        records.append({
            "chunk_id": f"{doc_id}::chunk::{kept}",
            "doc_id": doc_id,
            "chunk_index": kept,
            "chunk_type": ctype,
            "is_table": ctype == "table",
            "is_figure": ctype == "figure",
            "text": raw_text,
            "embed_text": embed_text,
            "token_count": chunker.tokenizer.count_tokens(embed_text),
            "section_path": " > ".join(str(h).strip() for h in headings if str(h).strip()),
            "pages": sorted(pages),
            "metadata": {},
        })
        kept += 1
    return records


def embed_fn_stub(texts: list[str]) -> list[list[float]]:
    """Deciding the embedding model is stage 03's job. This chunker only
    needs *something* pluggable here so the contract is complete end to end;
    swap this for a real embedding call (or the hash stub from notebook 01)
    without touching anything above."""
    raise NotImplementedError(
        "embedding is stage 03's job — plug a real embed_fn (or notebook 01's "
        "hash_embed_stub) in here if you need vectors, not chunk records"
    )


records = chunk_document_to_records(doc, doc_id="synthetic-outcomes-paper")
print(f"{len(records)} chunks\n")
for r in records:
    print(f"--- chunk {r['chunk_index']}  type={r['chunk_type']}  "
          f"tokens={r['token_count']}  section={r['section_path']!r} ---")
    print(r["text"])
    print()


## Step 12 — overlap in effect: chunk N's `embed_text` now carries chunk N-1's tail

With `overlap=0` (the call above), `records[1]['embed_text']` is exactly `HybridChunker`'s own text for that chunk -- nothing else. Re-run with `overlap=15` and the same chunk's `embed_text` should now start with the last ~15 tokens of chunk 0, exactly as `_tail_by_tokens` produced them in Step 10. This is the one property the port from `docling/chunking.py` exists to restore -- without it, this notebook's chunks had zero overlap, a different chunker from the one it was ported from, presented as the same thing.

In [ ]:
records_overlap = chunk_document_to_records(doc, doc_id="synthetic-outcomes-paper", overlap=15)

no_overlap_text = records[1]["embed_text"]
with_overlap_text = records_overlap[1]["embed_text"]

print("chunk 1, overlap=0:")
print(repr(no_overlap_text[:120]))
print()
print("chunk 1, overlap=15:")
print(repr(with_overlap_text[:160]))

assert with_overlap_text != no_overlap_text, "overlap=15 must change embed_text"
assert with_overlap_text.endswith(no_overlap_text), "overlap must PREPEND, not replace, the chunk's own text"
print()
print("confirmed: overlap=15 prepends chunk 0's tail without altering chunk 1's own content")

## The contrast — this is the entire point of the stage

Notebook `01`'s character splitter cut the outcomes table across a chunk
boundary and stranded the column header in the earlier chunk. Here, the
identical table (same rows, same header) survives as a **single chunk**,
tagged `chunk_type="table"`, with its markdown header row intact — because
`HybridChunker` walks Docling's structured document tree instead of a flat
word count, and a `TableItem` is one indivisible unit in that tree.

Check the cell above: exactly one record has `chunk_type == "table"`, and its
`text` is a complete five-row markdown table with the header row present —
not a fragment.

In [ ]:
table_records = [r for r in records if r["chunk_type"] == "table"]
assert len(table_records) == 1, "expected exactly one table chunk"
assert "Patient ID" in table_records[0]["text"], "table header must survive intact"
assert "P005" in table_records[0]["text"], "table must include its last row"
print("Table survived as one chunk, header and all:")
print(table_records[0]["text"])
